In [ ]:
# In[1]:

import pandas as pd

# helper to process each file
def summarize_file(fname, name_col):
    # read CSV
    df = pd.read_csv(fname)
    # parse timestamp
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    # row count
    row_count = int(len(df))
    # earliest / latest timestamps as ISO strings (UTC) or None if no timestamps
    if row_count > 0 and 'timestamp' in df.columns:
        earliest = df['timestamp'].min().isoformat()
        latest = df['timestamp'].max().isoformat()
    else:
        earliest = None
        latest = None
    # top cmdb_id values
    if 'cmdb_id' in df.columns and row_count > 0:
        vc = df['cmdb_id'].value_counts().head(20)
        top_cmdb = {str(k): int(v) for k, v in vc.items()}
    else:
        top_cmdb = {}
    # top name/message values
    if name_col is not None and name_col in df.columns and row_count > 0:
        vc2 = df[name_col].value_counts().head(20)
        top_names = {str(k): int(v) for k, v in vc2.items()}
    else:
        top_names = {}
    return {
        "row_count": row_count,
        "earliest_timestamp_utc": earliest,
        "latest_timestamp_utc": latest,
        "top_cmdb_id_counts": top_cmdb,
        "top_name_counts": top_names
    }

# process each file according to specified name column
summary = {
    "metric.csv": summarize_file("metric.csv", "kpi_name"),
    "trace.csv": summarize_file("trace.csv", "trace_name"),
    "log.csv": summarize_file("log.csv", "log_name"),
    "error_logs.csv": summarize_file("error_logs.csv", "message"),
}

summary

```
Out[1]:
```
```python
# Summarize the previously computed `summary` dict in plain English.
# Reuse the existing `summary` variable from earlier execution.
s = summary  # reuse variable

result = (
    f"metric.csv: {s['metric.csv']['row_count']} rows, time range {s['metric.csv']['earliest_timestamp_utc']} to {s['metric.csv']['latest_timestamp_utc']}. "
    f"Top services (cmdb_id) by row count: {', '.join([f'{k} ({v})' for k,v in list(s['metric.csv']['top_cmdb_id_counts'].items())[:6]])}. "
    f"Top KPIs: {', '.join([f'{k} ({v})' for k,v in list(s['metric.csv']['top_name_counts'].items())[:8]])}.\n\n"
    f"trace.csv: {s['trace.csv']['row_count']} rows (no trace data available).\n\n"
    f"log.csv: {s['log.csv']['row_count']} rows, time range {s['log.csv']['earliest_timestamp_utc']} to {s['log.csv']['latest_timestamp_utc']}. "
    f"Top services (cmdb_id): {', '.join([f'{k} ({v})' for k,v in list(s['log.csv']['top_cmdb_id_counts'].items())[:6]])}. "
    f"Top log names: {', '.join([f'{k} ({v})' for k,v in list(s['log.csv']['top_name_counts'].items())[:5]])}.\n\n"
    f"error_logs.csv: {s['error_logs.csv']['row_count']} rows (no error log messages available)."
)

result
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'row_count': 1882, 'earliest_timestamp_utc': '2024-01-18T22:09:00+00:00', 'latest_timestamp_utc': '2024-01-18T22:33:00+00:00', 'top_cmdb_id_counts': {'carts': 169, 'catalogue': 169, 'front-end': 168, 'orders': 159, 'payment': 159, 'user': 149, 'shipping': 134, 'orders-db': 100, 'catalogue-db': 100, 'carts-db': 100, 'queue-master': 100, 'session-db': 100, 'rabbitmq': 100, 'user-db': 100, 'rabbitmq-exporter': 75}, 'top_name_counts': {'socket': 375, 'mem': 375, 'cpu': 374, 'workload': 175, 'diskio': 175, 'error': 138, 'latency-90': 135, 'latency-50': 135}}, 'trace.csv': {'row_count': 0, 'earliest_timestamp_utc': None, 'latest_timestamp_utc': None, 'top_cmdb_id_counts': {}, 'top_name_counts': {}}, 'log.csv': {'row_count': 360, 'earliest_timestamp_utc': '2024-01-18T22:09:00+00:00', 'latest_timestamp_utc': '2024-01-18T22:32:00+00:00', 'top_cmdb_id_counts': {'carts-db': 48, 'catalogue': 48, 'payment': 48, 'front-end': 42, 'user': 40, 'queue-master': 30, 'orders': 30, 'shipping': 30, 'carts': 24, 'session-db': 10, 'user-db': 10}, 'top_name_counts': {'log.error_count': 180, 'log.row_count': 180}}, 'error_logs.csv': {'row_count': 0, 'earliest_timestamp_utc': None, 'latest_timestamp_utc': None, 'top_cmdb_id_counts': {}, 'top_name_counts': {}}}```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing variable names if present; otherwise load files
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")

# parse timestamps as UTC datetimes
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# candidate services and incident window (UTC)
candidates = ['carts','catalogue','orders','payment','user']
window_start = pd.Timestamp("2024-01-18T22:06:00", tz='UTC')
window_end = pd.Timestamp("2024-01-18T22:36:00", tz='UTC')

# Filter metric to candidate services
metric_candidates = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Compute global P95 and P5 for each (cmdb_id, kpi_name) pair using the full series
grp = metric_candidates.groupby(['cmdb_id', 'kpi_name'], sort=True)

thresholds = grp['value'].agg(
    global_p95=lambda s: float(s.dropna().quantile(0.95)) if s.dropna().shape[0] > 0 else None,
    global_p5 = lambda s: float(s.dropna().quantile(0.05)) if s.dropna().shape[0] > 0 else None,
    total_points = lambda s: int(s.shape[0])
).reset_index()

# Filter metric points to incident window (inclusive)
metric_window = metric_candidates[(metric_candidates['timestamp'] >= window_start) & (metric_candidates['timestamp'] <= window_end)].copy()

def earliest_iso_or_none(ts_series):
    if ts_series.empty or ts_series.isna().all():
        return None
    return pd.to_datetime(ts_series.min()).isoformat()

rows = []
for _, row in thresholds.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['global_p95']
    p5 = row['global_p5']
    w = metric_window[(metric_window['cmdb_id'] == cmdb) & (metric_window['kpi_name'] == kpi)]
    window_point_count = int(w.shape[0])
    if p95 is None:
        count_ge_p95 = 0
        earliest_ge_p95 = None
    else:
        ge = w[w['value'] >= p95]
        count_ge_p95 = int(ge.shape[0])
        earliest_ge_p95 = earliest_iso_or_none(ge['timestamp'])
    if p5 is None:
        count_le_p5 = 0
        earliest_le_p5 = None
    else:
        le = w[w['value'] <= p5]
        count_le_p5 = int(le.shape[0])
        earliest_le_p5 = earliest_iso_or_none(le['timestamp'])
    max_in_window = float(w['value'].max()) if window_point_count > 0 else None
    min_in_window = float(w['value'].min()) if window_point_count > 0 else None

    rows.append({
        "cmdb_id": cmdb,
        "kpi_name": kpi,
        "global_p95": p95,
        "global_p5": p5,
        "window_point_count": window_point_count,
        "count_values_ge_p95": count_ge_p95,
        "earliest_timestamp_ge_p95": earliest_ge_p95,
        "count_values_le_p5": count_le_p5,
        "earliest_timestamp_le_p5": earliest_le_p5,
        "max_value_in_window": max_in_window,
        "min_value_in_window": min_in_window
    })

metric_result_df = pd.DataFrame(rows).head(100)

# -----------------------
# Process log.csv for the same candidates and window
log_names_of_interest = ['log.error_count', 'log.row_count']
log_window = log_df[
    (log_df['cmdb_id'].isin(candidates)) &
    (log_df['timestamp'] >= window_start) &
    (log_df['timestamp'] <= window_end) &
    (log_df['log_name'].isin(log_names_of_interest))
].copy()

log_window['value'] = pd.to_numeric(log_window['value'], errors='coerce')

# Use group keys from groupby.groups to avoid calling builtins that might be shadowed
log_groups = log_window.groupby('cmdb_id', sort=True)
group_keys = set(log_groups.groups.keys())

log_rows = []
for cmdb in candidates:
    if cmdb in group_keys:
        grp_df = log_groups.get_group(cmdb)
    else:
        grp_df = pd.DataFrame(columns=log_window.columns)
    row_count_in_window = int(grp_df.shape[0])
    error_df = grp_df[grp_df['log_name'] == 'log.error_count']
    sum_log_error_count = float(error_df['value'].sum()) if not error_df.empty else 0.0
    max_log_error_count = float(error_df['value'].max()) if not error_df.empty else None
    gt0 = error_df[error_df['value'] > 0]
    earliest_ts_gt0 = pd.to_datetime(gt0['timestamp'].min()).isoformat() if not gt0.empty else None

    log_rows.append({
        "cmdb_id": cmdb,
        "row_count_in_window": row_count_in_window,
        "sum_log_error_count_in_window": sum_log_error_count,
        "max_log_error_count_in_window": max_log_error_count,
        "earliest_timestamp_with_log_error_count_gt0": earliest_ts_gt0
    })

log_result_df = pd.DataFrame(log_rows)

# Convert DataFrames to JSON-serializable records, replacing NaN with None
def df_to_records_clean(df):
    return df.where(pd.notnull(df), None).to_dict(orient='records')

result = {
    "metric_summary": df_to_records_clean(metric_result_df),
    "log_summary": df_to_records_clean(log_result_df)
}

result

```
Out[2]:
```
Summary of findings (incident window 2024-01-18 22:06–22:36 UTC):

Metric highlights by service (only KPIs with notable events shown):
- carts
  - Multiple KPIs (cpu, mem, socket, workload, latency-50, latency-90) had values >= the global P95 during the window (typically 2 points each). Earliest exceedances around 22:10–22:18 (e.g., cpu at 22:10). Error KPI was 0.0 across the window.
  - Typical window point counts: 22–25 per KPI. Max cpu ~1.51, max socket 19.2, max workload ~8.66.

- catalogue
  - socket is constant at 7.0 (count_ge_p95 = 25 because global P95 = 7.0). Other KPIs (cpu, mem, latency) show a few values >= global P95 (earliest around 22:15–22:25).
  - Error KPI = 0.0 across the window.

- orders
  - cpu, mem, socket, workload showed values >= global P95 (a few points each; earliest cpu exceedance 22:10).
  - The orders.error KPI had small non-zero values: global_p95 ≈ 0.0213 and two window points >= p95 with earliest at 22:27; max error in window ≈ 0.1067.
  - Some latency KPIs had high values (latency-50 and latency-90 showing single exceedances around 22:26–22:28).

- payment
  - cpu, mem, workload, socket had some points >= global P95 (earliest around 22:11 or later). Error KPI = 0.0 across the window.
  - socket constant at 5.0 (count_ge_p95 = 25).

- user
  - Several KPIs (cpu, mem, socket, workload, latency) show exceedances of global P95 (usually 1–2 points). Earliest socket exceedance at 22:28; user.error had one point >= global_p95 at 22:28 (max user.error in window ≈ 0.361).
  - Max socket ~34.32 and max workload ~18.92 in-window.

Log summary (same window, candidate services):
- carts: 24 log rows, sum(log.error_count)=0.0, no log.error_count > 0 timestamps.
- catalogue: 48 log rows, sum(log.error_count)=0.0, no log.error_count > 0 timestamps.
- orders: 30 log rows, sum(log.error_count)=0.0, no log.error_count > 0 timestamps.
- payment: 48 log rows, sum(log.error_count)=0.0, no log.error_count > 0 timestamps.
- user: 40 log rows, sum(log.error_count)=0.0, no log.error_count > 0 timestamps.

Concise interpretation:
- Several performance-related KPIs (cpu, mem, socket, workload, latencies) across the candidate services had a small number of values exceeding their global P95 during the incident window; these exceedances occur at various times between ~22:09 and ~22:28 depending on service/KPI.
- Normalized log error counts (log.error_count) show zero in the window for all candidate services despite some metric-level error KPIs (orders.error and user.error) showing small non-zero values.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric_summary': [{'cmdb_id': 'carts', 'kpi_name': 'cpu', 'global_p95': 1.4766269791046391, 'global_p5': 0.14109454097045593, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:10:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:32:00+00:00', 'max_value_in_window': 1.5126654396389467, 'min_value_in_window': 0.1359320821179043}, {'cmdb_id': 'carts', 'kpi_name': 'error', 'global_p95': 0.0, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 25, 'earliest_timestamp_ge_p95': '2024-01-18T22:09:00+00:00', 'count_values_le_p5': 25, 'earliest_timestamp_le_p5': '2024-01-18T22:09:00+00:00', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'carts', 'kpi_name': 'latency-50', 'global_p95': 0.058922092318256125, 'global_p5': 0.03582695360326315, 'window_point_count': 22, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:23:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:28:00+00:00', 'max_value_in_window': 0.0600019409937888, 'min_value_in_window': 0.025}, {'cmdb_id': 'carts', 'kpi_name': 'latency-90', 'global_p95': 0.23768908398573052, 'global_p5': 0.12134306104088446, 'window_point_count': 22, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:22:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:10:00+00:00', 'max_value_in_window': 0.2813310344827587, 'min_value_in_window': 0.0994444444444444}, {'cmdb_id': 'carts', 'kpi_name': 'mem', 'global_p95': 217088382.29333332, 'global_p5': 217063478.6133333, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:16:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:32:00+00:00', 'max_value_in_window': 217100765.86666667, 'min_value_in_window': 217063424.0}, {'cmdb_id': 'carts', 'kpi_name': 'socket', 'global_p95': 19.12333333333333, 'global_p5': 13.02, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:12:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:32:00+00:00', 'max_value_in_window': 19.2, 'min_value_in_window': 13.0}, {'cmdb_id': 'carts', 'kpi_name': 'workload', 'global_p95': 8.372306666666667, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:18:00+00:00', 'count_values_le_p5': 3, 'earliest_timestamp_le_p5': '2024-01-18T22:31:00+00:00', 'max_value_in_window': 8.66005, 'min_value_in_window': 0.0}, {'cmdb_id': 'catalogue', 'kpi_name': 'cpu', 'global_p95': 0.18163858259575447, 'global_p5': 0.03365101241399272, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:15:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:22:00+00:00', 'max_value_in_window': 0.1956358592811201, 'min_value_in_window': 0.0314892061924639}, {'cmdb_id': 'catalogue', 'kpi_name': 'error', 'global_p95': 0.0, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 25, 'earliest_timestamp_ge_p95': '2024-01-18T22:09:00+00:00', 'count_values_le_p5': 25, 'earliest_timestamp_le_p5': '2024-01-18T22:09:00+00:00', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'catalogue', 'kpi_name': 'latency-50', 'global_p95': 0.0030858176324679446, 'global_p5': 0.0030146777475530003, 'window_point_count': 22, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:25:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:23:00+00:00', 'max_value_in_window': 0.0030940656565656, 'min_value_in_window': 0.0030040846633226}, {'cmdb_id': 'catalogue', 'kpi_name': 'latency-90', 'global_p95': 0.004754471738442345, 'global_p5': 0.0046264199455954445, 'window_point_count': 22, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:25:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:23:00+00:00', 'max_value_in_window': 0.0047693181818181, 'min_value_in_window': 0.0046073523939808}, {'cmdb_id': 'catalogue', 'kpi_name': 'mem', 'global_p95': 6387493.546666667, 'global_p5': 5989471.573333333, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:15:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:32:00+00:00', 'max_value_in_window': 6406621.866666666, 'min_value_in_window': 5722112.0}, {'cmdb_id': 'catalogue', 'kpi_name': 'socket', 'global_p95': 7.0, 'global_p5': 7.0, 'window_point_count': 25, 'count_values_ge_p95': 25, 'earliest_timestamp_ge_p95': '2024-01-18T22:09:00+00:00', 'count_values_le_p5': 25, 'earliest_timestamp_le_p5': '2024-01-18T22:09:00+00:00', 'max_value_in_window': 7.0, 'min_value_in_window': 7.0}, {'cmdb_id': 'catalogue', 'kpi_name': 'workload', 'global_p95': 4.19846, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:18:00+00:00', 'count_values_le_p5': 3, 'earliest_timestamp_le_p5': '2024-01-18T22:31:00+00:00', 'max_value_in_window': 4.2822000000000005, 'min_value_in_window': 0.0}, {'cmdb_id': 'orders', 'kpi_name': 'cpu', 'global_p95': 1.0501781402788688, 'global_p5': 0.13270786670638923, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:10:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:31:00+00:00', 'max_value_in_window': 1.0568277515759454, 'min_value_in_window': 0.1302750364568033}, {'cmdb_id': 'orders', 'kpi_name': 'error', 'global_p95': 0.021333333333333204, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:27:00+00:00', 'count_values_le_p5': 23, 'earliest_timestamp_le_p5': '2024-01-18T22:09:00+00:00', 'max_value_in_window': 0.1066833333333333, 'min_value_in_window': 0.0}, {'cmdb_id': 'orders', 'kpi_name': 'latency-50', 'global_p95': 0.7214447862079442, 'global_p5': 0.03860150488676372, 'window_point_count': 17, 'count_values_ge_p95': 1, 'earliest_timestamp_ge_p95': '2024-01-18T22:28:00+00:00', 'count_values_le_p5': 1, 'earliest_timestamp_le_p5': '2024-01-18T22:10:00+00:00', 'max_value_in_window': 0.7283260233918128, 'min_value_in_window': 0.0382028037898886}, {'cmdb_id': 'orders', 'kpi_name': 'latency-90', 'global_p95': 2.267982270050748, 'global_p5': 0.05244076875296636, 'window_point_count': 17, 'count_values_ge_p95': 1, 'earliest_timestamp_ge_p95': '2024-01-18T22:26:00+00:00', 'count_values_le_p5': 1, 'earliest_timestamp_le_p5': '2024-01-18T22:10:00+00:00', 'max_value_in_window': 2.2814959775829338, 'min_value_in_window': 0.048776430750371}, {'cmdb_id': 'orders', 'kpi_name': 'mem', 'global_p95': 331382319.7866667, 'global_p5': 331198464.0, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:26:00+00:00', 'count_values_le_p5': 3, 'earliest_timestamp_le_p5': '2024-01-18T22:23:00+00:00', 'max_value_in_window': 331457604.26666665, 'min_value_in_window': 331198464.0}, {'cmdb_id': 'orders', 'kpi_name': 'socket', 'global_p95': 16.61, 'global_p5': 7.0, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:16:00+00:00', 'count_values_le_p5': 8, 'earliest_timestamp_le_p5': '2024-01-18T22:23:00+00:00', 'max_value_in_window': 18.75, 'min_value_in_window': 7.0}, {'cmdb_id': 'orders', 'kpi_name': 'workload', 'global_p95': 2.09241, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:16:00+00:00', 'count_values_le_p5': 9, 'earliest_timestamp_le_p5': '2024-01-18T22:22:00+00:00', 'max_value_in_window': 2.1067, 'min_value_in_window': 0.0}, {'cmdb_id': 'payment', 'kpi_name': 'cpu', 'global_p95': 0.08184776490879349, 'global_p5': 0.03464395009355182, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:11:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:30:00+00:00', 'max_value_in_window': 0.0842345488823286, 'min_value_in_window': 0.032610796419633}, {'cmdb_id': 'payment', 'kpi_name': 'error', 'global_p95': 0.0, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 25, 'earliest_timestamp_ge_p95': '2024-01-18T22:09:00+00:00', 'count_values_le_p5': 25, 'earliest_timestamp_le_p5': '2024-01-18T22:09:00+00:00', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'payment', 'kpi_name': 'latency-50', 'global_p95': 0.00215541239592388, 'global_p5': 0.00178162005362908, 'window_point_count': 17, 'count_values_ge_p95': 1, 'earliest_timestamp_ge_p95': '2024-01-18T22:12:00+00:00', 'count_values_le_p5': 1, 'earliest_timestamp_le_p5': '2024-01-18T22:28:00+00:00', 'max_value_in_window': 0.0021958592097354, 'min_value_in_window': 0.0012211316509166}, {'cmdb_id': 'payment', 'kpi_name': 'latency-90', 'global_p95': 0.004443393485804661, 'global_p5': 0.004355951886543199, 'window_point_count': 17, 'count_values_ge_p95': 1, 'earliest_timestamp_ge_p95': '2024-01-18T22:13:00+00:00', 'count_values_le_p5': 1, 'earliest_timestamp_le_p5': '2024-01-18T22:28:00+00:00', 'max_value_in_window': 0.0044602800612353, 'min_value_in_window': 0.0042479105407096}, {'cmdb_id': 'payment', 'kpi_name': 'mem', 'global_p95': 4514720.426666667, 'global_p5': 4052855.466666667, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:21:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:32:00+00:00', 'max_value_in_window': 4515908.266666667, 'min_value_in_window': 4022272.0}, {'cmdb_id': 'payment', 'kpi_name': 'socket', 'global_p95': 5.0, 'global_p5': 5.0, 'window_point_count': 25, 'count_values_ge_p95': 25, 'earliest_timestamp_ge_p95': '2024-01-18T22:09:00+00:00', 'count_values_le_p5': 25, 'earliest_timestamp_le_p5': '2024-01-18T22:09:00+00:00', 'max_value_in_window': 5.0, 'min_value_in_window': 5.0}, {'cmdb_id': 'payment', 'kpi_name': 'workload', 'global_p95': 2.09832, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:15:00+00:00', 'count_values_le_p5': 9, 'earliest_timestamp_le_p5': '2024-01-18T22:22:00+00:00', 'max_value_in_window': 2.1167333333333334, 'min_value_in_window': 0.0}, {'cmdb_id': 'user', 'kpi_name': 'cpu', 'global_p95': 1.0496051727696383, 'global_p5': 0.01021776281243266, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:09:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:23:00+00:00', 'max_value_in_window': 1.0737674281275502, 'min_value_in_window': 0.0094723028540156}, {'cmdb_id': 'user', 'kpi_name': 'error', 'global_p95': 0.1977333333333329, 'global_p5': 0.0, 'window_point_count': 13, 'count_values_ge_p95': 1, 'earliest_timestamp_ge_p95': '2024-01-18T22:28:00+00:00', 'count_values_le_p5': 11, 'earliest_timestamp_le_p5': '2024-01-18T22:21:00+00:00', 'max_value_in_window': 0.3610333333333333, 'min_value_in_window': 0.0}, {'cmdb_id': 'user', 'kpi_name': 'latency-50', 'global_p95': 0.23539393774990244, 'global_p5': 0.0042766414111230005, 'window_point_count': 18, 'count_values_ge_p95': 1, 'earliest_timestamp_ge_p95': '2024-01-18T22:22:00+00:00', 'count_values_le_p5': 1, 'earliest_timestamp_le_p5': '2024-01-18T22:15:00+00:00', 'max_value_in_window': 0.2554627584327639, 'min_value_in_window': 0.0042120547053049}, {'cmdb_id': 'user', 'kpi_name': 'latency-90', 'global_p95': 1.033648326710868, 'global_p5': 0.008746970628244255, 'window_point_count': 18, 'count_values_ge_p95': 1, 'earliest_timestamp_ge_p95': '2024-01-18T22:27:00+00:00', 'count_values_le_p5': 1, 'earliest_timestamp_le_p5': '2024-01-18T22:15:00+00:00', 'max_value_in_window': 1.2887310562313314, 'min_value_in_window': 0.0087222513807585}, {'cmdb_id': 'user', 'kpi_name': 'mem', 'global_p95': 11899289.6, 'global_p5': 4362076.16, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:24:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:32:00+00:00', 'max_value_in_window': 12172424.533333331, 'min_value_in_window': 3170304.0}, {'cmdb_id': 'user', 'kpi_name': 'socket', 'global_p95': 34.21333333333333, 'global_p5': 10.333333333333336, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:28:00+00:00', 'count_values_le_p5': 2, 'earliest_timestamp_le_p5': '2024-01-18T22:32:00+00:00', 'max_value_in_window': 34.31666666666667, 'min_value_in_window': 4.0}, {'cmdb_id': 'user', 'kpi_name': 'workload', 'global_p95': 18.85777333333333, 'global_p5': 0.0, 'window_point_count': 25, 'count_values_ge_p95': 2, 'earliest_timestamp_ge_p95': '2024-01-18T22:14:00+00:00', 'count_values_le_p5': 8, 'earliest_timestamp_le_p5': '2024-01-18T22:22:00+00:00', 'max_value_in_window': 18.91785, 'min_value_in_window': 0.0}], 'log_summary': [{'cmdb_id': 'carts', 'row_count_in_window': 24, 'sum_log_error_count_in_window': 0.0, 'max_log_error_count_in_window': 0.0, 'earliest_timestamp_with_log_error_count_gt0': None}, {'cmdb_id': 'catalogue', 'row_count_in_window': 48, 'sum_log_error_count_in_window': 0.0, 'max_log_error_count_in_window': 0.0, 'earliest_timestamp_with_log_error_count_gt0': None}, {'cmdb_id': 'orders', 'row_count_in_window': 30, 'sum_log_error_count_in_window': 0.0, 'max_log_error_count_in_window': 0.0, 'earliest_timestamp_with_log_error_count_gt0': None}, {'cmdb_id': 'payment', 'row_count_in_window': 48, 'sum_log_error_count_in_window': 0.0, 'max_log_error_count_in_window': 0.0, 'earliest_timestamp_with_log_error_count_gt0': None}, {'cmdb_id': 'user', 'row_count_in_window': 40, 'sum_log_error_count_in_window': 0.0, 'max_log_error_count_in_window': 0.0, 'earliest_timestamp_with_log_error_count_gt0': None}]}```
```